# Branch-And-Bound Variants Optimised for Gradient Sum


## Test Cases

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

In [2]:
from testdata import SMALL_1


## Use in Branch-and-Bound

In [3]:
import numpy as np
from optikon import compute_bounds, equal_width_propositionalization, make_maxheap_class
from testdata import mvn_with_correlation
from numba.experimental import jitclass
from numba import njit
from numba.types import int64, float64

node_spec = [
    ('key', int64[:]),
    ('critical', int64[:]),
    ('remaining', int64[:]),
    ('support', int64[:]),
    ('pos_support', int64[:]),
]

@jitclass(node_spec)
class Node:
    def __init__(self, key, critical, remaining, support, pos_support):
        self.key = key
        self.critical = critical
        self.remaining = remaining
        self.support = support
        self.pos_support = pos_support

@njit
def make_root(x, y, prop):
    l, u = compute_bounds(x)
    remaining = prop.nontrivial(l, u, np.arange(len(prop)))
    empty = np.empty(0, dtype=np.int64)
    support = np.arange(len(x))
    pos_support = support[y > 0]
    return Node(empty, empty, remaining, support, pos_support)

make_root(SMALL_1.x, SMALL_1.y, SMALL_1.prop)

NODE_TYPE = Node.class_type.instance_type  
NodeHeap = make_maxheap_class(float64, NODE_TYPE)
node_heap = NodeHeap()
node_heap.push(0, make_root(SMALL_1.x, SMALL_1.y, SMALL_1.prop))
node_heap.pop()

(0.0, <numba.experimental.jitclass.boxing.Node at 0x10591fca0>)

In [ ]:
from optikon import Propositionalization

@njit
def max_weighted_support(x, y, prop: Propositionalization, max_depth=4):
    heap = NodeHeap()

    root = make_root(x, y, prop)
    root_bound = y[root.pos_support].sum()
    root_value = y.sum()
    heap.push(root_bound, root)

    best_key = root.key
    best_val = root_value
    nodes_created = 1
    candidate_edges = 0

    while heap:
        key, node = heap.pop()
        
        if key <= best_val:
            break

        if len(node.key) >= max_depth:
            continue

        candidate_edges += len(node.remaining)
        for p_idx in range(len(node.remaining)):
            p = node.remaining[p_idx]

            _key = np.empty(len(node.key) + 1, dtype=np.int64)
            _key[:-1] = node.key
            _key[-1] = p

            _sup = node.support[prop.support_specific(x[node.support], p)]
            _pos_sup = node.pos_support[prop.support_specific(x[node.pos_support], p)]

            _val = y[_sup].sum()
            _bound = y[_pos_sup].sum()

            if _val > best_val:
                best_val = _val
                best_key = _key

            if _bound <= best_val:
                continue

            _crit = np.empty(len(node.critical) + p_idx, dtype=np.int64)
            _crit[:len(node.critical)] = node.critical
            _crit[len(node.critical):] = node.remaining[:p_idx]

            l, u = compute_bounds(x[_sup])
            if len(prop.trivial(l, u, _crit)) > 0:
                continue

            _rem = prop.nontrivial(l, u, node.remaining[p_idx+1:])

            heap.push(_bound, Node(_key, _crit, _rem, _sup, _pos_sup))

            nodes_created += 1

    return best_key, best_val, nodes_created, candidate_edges

key, val, created, candidate_edges = max_weighted_support(SMALL_1.x, SMALL_1.y, SMALL_1.prop)
key, SMALL_1.prop.str_from_conj(key), val, created, candidate_edges
key, SMALL_1.prop.str_from_conj(key), val, created, candidate_edges

(array([ 3, 19]), 'x1 >= 1.000 & x3 >= 1.000', 3, 4, 43)

In [8]:
SMALL_1.prop[key].support_all(SMALL_1.x)

array([1, 2, 4])

In [5]:
x = mvn_with_correlation(200, seed=0)
y = np.random.default_rng(seed=0).normal(size=200)
prop = equal_width_propositionalization(x)

key, val, created, candidate_edges = max_weighted_support(x, y, prop, 8)
prop.str_from_conj(key), val, created, candidate_edges

('x1 >= -1.322 & x1 <= 1.747 & x2 <= 1.413 & x3 <= 1.941 & x4 <= 0.557',
 25.699605891610826,
 385690,
 1253849)

In [6]:
%timeit max_weighted_support(x, y, prop, 8)

3.65 s ± 25.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
prop.support_all(x, key)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 13, 14, 15, 20, 21, 23, 24,
       28, 31, 32, 35, 36, 37, 38, 39, 40, 45, 46, 47, 49, 50, 53, 55, 56,
       57, 62, 65, 66, 67, 68, 70, 72, 74, 76, 77, 79, 80, 82, 85, 86, 88,
       89, 91, 92, 95, 96, 97, 98])

(array([ 8, 16, 23, 49, 59, 66]),
 'x1 >= -1.472 & x1 <= 1.740 & x2 >= 0.612 & x3 <= 1.730 & x4 >= -2.580 & x4 <= 0.538',
 20.397495605985817)

In [ ]:
prop.support_all(x, np.array([ 8, 16, 23, 49, 59, 66]))

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 13, 14, 15, 20, 21, 23, 24,
       28, 31, 32, 35, 36, 37, 38, 39, 40, 45, 46, 47, 49, 50, 53, 55, 56,
       57, 62, 65, 66, 67, 68, 70, 72, 74, 76, 77, 79, 80, 82, 85, 86, 88,
       89, 91, 92, 95, 96, 97, 98])

In [ ]:
np.array_equal(prop.support_all(x, key), prop.support_all(x, np.array([ 8, 16, 23, 49, 59, 66])))

True